# Résumé de ce qu'on a fait

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram
import plotly.graph_objects as go
import os
import warnings
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import accuracy_score
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)
from sklearn.preprocessing import LabelEncoder

In [2]:
li_filenames = []

for racine, _, fichiers in os.walk('cross-era_chroma-nnls'):
    for fichier in fichiers:
        chemin_relatif = os.path.relpath(os.path.join(racine, fichier))
        li_filenames.append(chemin_relatif)

In [3]:
def periode(str):
    try:
        date_deb = int(str[0:4])
        date_end = str[5:9]
        if date_deb is None :
            return "Unknown"
        elif date_deb < 1570 and date_deb > 476:
            return "Medieval"
        elif date_deb < 1730:
            return "Baroque"
        elif date_deb < 1800:
            return "Classique"
        elif date_deb < 1860:
            return "Romantique"
        else:
            try:
                date_end = int(date_end)
                return "XXe"
            except:
                return "contemporain"
    except :
        return "Unknown"

In [4]:
def dico_filenames(filename = "cross-era_annotations.csv"):
    dico = {}
    dico2 = {}
    df = pd.read_csv(filename, sep=',')
    for i in range(len(df)):
        dico[df['Filename'][i]] = df['Composer'][i]
        dico2[df['Filename'][i]] = periode(df['CompLifetime'][i])
    return dico, dico2

In [6]:
pd.read_csv('cross-era_annotations.csv', sep=',')

,Class,Filename,CrossEra-ID,Instrumentation,Key,Mode,Composer,CompLifetime,Country,Unnamed: 9
0,orchestra_baroque,CrossEra-0001_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0001,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
1,orchestra_baroque,CrossEra-0002_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0002,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
2,orchestra_baroque,CrossEra-0003_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0003,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
3,orchestra_baroque,CrossEra-0004_Albinoni__sinata_a_cinque_no._6_...,CrossEra-0004,orchestra,G,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
4,orchestra_baroque,CrossEra-0005_Albinoni_concerto_in_a_minor_bwv...,CrossEra-0005,orchestra,A,minor,Albinoni; Tomaso,1671-1751,Italy,NaN
...,...,...,...,...,...,...,...,...,...,...
1995,piano_addon,CrossEra-1996_Weber_sonata_no._35_in_a_minor_o...,CrossEra-1996,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1996,piano_addon,CrossEra-1997_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1997,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1997,piano_addon,CrossEra-1998_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1998,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN
1998,piano_addon,CrossEra-1999_Weber_sonata_no._4_in_e_minor_op...,CrossEra-1999,piano,NaN,NaN,Weber; Carl Maria von,1786-1826,Germany,NaN


In [5]:
dico_composers, dico_periodes = dico_filenames()
a = np.array(list(dico_composers.values()))
np.unique(a)
# Imprime toutes les clés ayant pour valeur 'major'
for key, value in dico_composers.items():
    if value == ' major':
        print(key)
dico_composers["CrossEra-0616_Borodin_symphony_no.3_in_a_minor_moderato_assai.mp3"] = 'Borodin; Alexander'
dico_composers["CrossEra-0673_Liszt_poems__mazeppa.mp3"] = 'Liszt; Franz'
dico_composers["CrossEra-0674_Liszt_poems__prometheus.mp3"] = 'Liszt; Franz'
dico_composers["CrossEra-0776_Verdi_Overt_giovanna_darco_sinfonia.mp3"] = 'Verdi; Giuseppe'
dico_composers["CrossEra-1016_Cimarosa_Piano_sonata_no._24_in_b-flat_minor_major___andantino.mp3"] = 'Cimarosa; Domenico'

CrossEra-0616_Borodin_symphony_no.3_in_a_minor_moderato_assai.mp3
CrossEra-0673_Liszt_poems__mazeppa.mp3
CrossEra-0674_Liszt_poems__prometheus.mp3
CrossEra-0776_Verdi_Overt_giovanna_darco_sinfonia.mp3
CrossEra-1016_Cimarosa_Piano_sonata_no._24_in_b-flat_minor_major___andantino.mp3


In [6]:
# Load the data
with open("hist_cross-era.pkl", "rb") as f:
    dico_cross_era = pickle.load(f)

In [7]:
X = []
etiquettes = []
y = []
periode = []
for file, histograms in dico_cross_era.items():
    composer = dico_composers[file.split('/')[1]]
    etiquettes.append(file)
    matrice = np.vstack(histograms)
    X.append(matrice.flatten())
    y.append(composer)
    periode.append(dico_periodes[file.split('/')[1]])
X = np.array(X)
X_scaled = StandardScaler().fit_transform(X)
y = np.array(y)
periodes = np.array(periode)
morceaux = np.array(etiquettes)

In [8]:
# Compter le nombre d'oeuvres par compositeur
oeuvres_par_compositeur = pd.Series(y).value_counts()
oeuvres_par_compositeur

Bach; Johann Sebastian      120
Mozart; Wolfgang Amadeus    114
Haydn; Joseph               100
Shostakovich; Dmitri         82
Beethoven; Ludwig van        62
                           ... 
Antheil; George               7
Rimsky-Korsakov; Nicolai      7
Berlioz; Hector               7
Ives; Charles Edward          6
Smetana; Bedrich              6
Name: count, Length: 70, dtype: int64

# Première analyse des données

In [11]:
import umap.umap_ as umap
from sklearn.preprocessing import LabelEncoder

reducer_3d = umap.UMAP(n_neighbors=4, min_dist=0.0, n_components=3, metric='cosine')
le = LabelEncoder()
y_numeric = le.fit_transform(y)
X_umap_3d = reducer_3d.fit_transform(X_scaled, y=y_numeric)

fig = px.scatter_3d(
    x=X_umap_3d[:, 0],
    y=X_umap_3d[:, 1],
    z=X_umap_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection UMAP 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()


In [12]:
import umap.umap_ as umap
from sklearn.preprocessing import LabelEncoder
fig = px.scatter_3d(
    x=X_umap_3d[:, 0],
    y=X_umap_3d[:, 1],
    z=X_umap_3d[:, 2],
    color=periodes,
    labels={'color': 'Période'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection UMAP 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [13]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_umap_3d)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = centroides.values

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs après UMAP",
    labels={'x': 'UMAP1', 'y': 'UMAP2', 'z': 'UMAP3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

In [14]:
from sklearn.manifold import TSNE

# Projection T-SNE 3D
tsne_3d = TSNE(n_components=3, perplexity=100, random_state=42, metric='cosine')
X_tsne_3d = tsne_3d.fit_transform(X_scaled)

fig = px.scatter_3d(
    x=X_tsne_3d[:, 0],
    y=X_tsne_3d[:, 1],
    z=X_tsne_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    title="Projection T-SNE 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [15]:
fig = px.scatter_3d(
    x=X_tsne_3d[:, 0],
    y=X_tsne_3d[:, 1],
    z=X_tsne_3d[:, 2],
    color=periodes,
    labels={'color': 'Période'},
    hover_name=morceaux,
    title="Projection T-SNE 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [16]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_tsne_3d)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = centroides.values

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs après T-SNE",
    labels={'x': 'x', 'y': 'y', 'z': 'z', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

In [17]:
from sklearn.decomposition import PCA

# PCA 3D
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_scaled)

fig = px.scatter_3d(
    x=X_pca_3d[:, 0],
    y=X_pca_3d[:, 1],
    z=X_pca_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection PCA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [18]:
fig = px.scatter_3d(
    x=X_pca_3d[:, 0],
    y=X_pca_3d[:, 1],
    z=X_pca_3d[:, 2],
    color=periodes,
    labels={'color': 'Période'},
    hover_name=periodes,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection PCA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [19]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_scaled)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = pca_3d.transform(centroides.values)

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs après PCA",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

In [20]:
# PCA réduction à 500 dimensions
pca_500 = PCA(n_components=1000)
X_pca_500 = pca_500.fit_transform(X_scaled)

# UMAP sur la réduction PCA
reducer_pca_umap = umap.UMAP(n_neighbors=10, min_dist=0.0, n_components=3, metric='cosine')
X_pca_umap_3d = reducer_pca_umap.fit_transform(X_pca_500, y=y_numeric)

fig = px.scatter_3d(
    x=X_pca_umap_3d[:, 0],
    y=X_pca_umap_3d[:, 1],
    z=X_pca_umap_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    title="PCA (500) + UMAP 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [21]:
fig = px.scatter_3d(
    x=X_pca_umap_3d[:, 0],
    y=X_pca_umap_3d[:, 1],
    z=X_pca_umap_3d[:, 2],
    color=periodes,
    labels={'color': 'Période'},
    hover_name=morceaux,
    title="PCA (500) + UMAP 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [22]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_pca_umap_3d)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = centroides.values

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (PCA 3D, coloré par période)",
    labels={'x': 'PCA1', 'y': 'PCA2', 'z': 'PCA3', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

## Clustering

In [10]:
lda_3d = LDA(n_components=3)
X_lda_3d = lda_3d.fit_transform(X_scaled, y)

fig = px.scatter_3d(
    x=X_lda_3d[:, 0],
    y=X_lda_3d[:, 1],
    z=X_lda_3d[:, 2],
    color=y,
    labels={'color': 'Compositeur'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection LDA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [11]:
fig = px.scatter_3d(
    x=X_lda_3d[:, 0],
    y=X_lda_3d[:, 1],
    z=X_lda_3d[:, 2],
    color=periodes,
    labels={'color': 'Période'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="Projection LDA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [12]:
# Calculer les centroïdes pour chaque compositeur
# On retire la colonne 'periode' avant de faire la moyenne
df_X = pd.DataFrame(X_scaled)
df_X['composer'] = y
centroides = df_X.groupby('composer').mean()
df_X['periode'] = periodes

# Récupérer la période dominante pour chaque compositeur
periode_centroide = df_X.groupby('composer')['periode'].agg(lambda x: x.value_counts().idxmax())

# Affichage des centroïdes sur plotly (projection PCA 3D pour visualisation)
centroides_pca = lda_3d.transform(centroides.values)

fig = px.scatter_3d(
    x=centroides_pca[:, 0],
    y=centroides_pca[:, 1],
    z=centroides_pca[:, 2],
    # text=centroides.index,
    hover_name=centroides.index,
    color=periode_centroide,
    title="Centroïdes des compositeurs (LDA 3D, coloré par période)",
    labels={'x': 'x', 'y': 'y', 'z': 'z', 'color': 'Période'}
)
fig.update_traces(marker=dict(size=8))
fig.show()

## Test de l'espace de représentation

In [25]:
# Load the data
with open("dico_hist_supp.pkl", "rb") as f:
    dico_supp = pickle.load(f)

In [26]:
Xsup = []
etiquettessup = []
ysup = []
periodesup = []
for file, histograms in dico_supp.items():
    etiquettessup.append(file)
    matrice = np.vstack(histograms)
    Xsup.append(matrice.flatten())
    ysup.append(file)
    periodesup.append(file)
X_tot = np.vstack([X, Xsup])
y_tot = np.concatenate([y, ysup])
morceaux_tot = np.concatenate([morceaux, etiquettessup])
periodes_tot = np.concatenate([periodes, periodesup])
scaler = StandardScaler()
X_scaled_tot = scaler.fit_transform(X_tot)

In [27]:
lda_3d = LDA(n_components=3)
X_lda_3d_tot = lda_3d.fit_transform(X_scaled_tot, y_tot)

fig = px.scatter_3d(
    x=X_lda_3d_tot[:, 0],
    y=X_lda_3d_tot[:, 1],
    z=X_lda_3d_tot[:, 2],
    color=y_tot,
    labels={'color': 'Compositeur'},
    hover_name=morceaux_tot,
    hover_data={'Compositeur': y_tot, 'Période': periodes_tot},
    title="Projection LDA 3D des morceaux"
)
fig.update_traces(marker=dict(size=4))
buttons = [
    dict(label='Tout masquer',
         method='restyle',
         args=['visible', ['legendonly'] * len(fig.data)]),
    
    dict(label='Tout afficher',
         method='restyle',
         args=['visible', [True] * len(fig.data)])
]

fig.update_layout(
    updatemenus=[dict(type='buttons', showactive=True, buttons=buttons)]
)
fig.show()

## Clustering brut des compositeurs

In [20]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

le = LabelEncoder()
y_numeric = le.fit_transform(y)

# Appliquer KMeans avec 99 clusters
kmeans = KMeans(n_clusters=70, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Appliquer LDA pour la visualisation (3 composantes)
lda_kmeans = LDA(n_components=3)
X_lda_kmeans = lda_kmeans.fit_transform(X_scaled, y)

# Visualisation des clusters dans l'espace LDA
fig = px.scatter_3d(
    x=X_lda_kmeans[:, 0],
    y=X_lda_kmeans[:, 1],
    z=X_lda_kmeans[:, 2],
    color=clusters.astype(str),
    labels={'color': 'Cluster'},
    hover_name=morceaux,
    hover_data={'Compositeur': y, 'Période': periodes},
    title="KMeans (70 clusters) + LDA 3D"
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [18]:
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'vscode'  # ou 'vscode' si tu es dans VS Code
#from plotly.callbacks import CallbackClient
# Synchronisation des caméras entre les deux sous-figures
# Correction : retirer la parenthèse fermante en trop à la ligne 7
# (Il n'est pas nécessaire d'importer CallbackClient pour Plotly, et la synchronisation avancée des caméras nécessite du JS personnalisé, non du Python pur)
import plotly.graph_objects as go
def sync_cameras(fig):
    # Ajoute un callback JavaScript pour synchroniser les caméras
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                showactive=False,
                buttons=[
                    dict(
                        label="Synchroniser les vues",
                        method="relayout",
                        args=[
                            None,
                            {
                                "scene.camera": None,
                                "scene2.camera": None
                            }
                        ]
                    )
                ]
            )
        ]
    )
    fig['layout']['scene']['camera'] = dict()
    fig['layout']['scene2']['camera'] = dict()
    fig.show(config={
        'scrollZoom': True,
        'displayModeBar': True,
        'modeBarButtonsToAdd': [
            {
                'name': 'Sync Views',
                'icon': 'arrows-h',
                'click': """
                    function(gd) {
                        var scene = gd._fullLayout.scene._scene;
                        var scene2 = gd._fullLayout.scene2._scene;
                        scene2.setCamera(scene.getCamera());
                    }
                """
            }
        ]
    })

# Projection UMAP 3D des morceaux avec coloration par cluster KMeans
X_umap_clusters_3d = lda_3d.fit_transform(X_scaled,y)

# Affichage côte à côte : clusters à gauche, compositeurs à droite

fig_sub = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
    subplot_titles=("UMAP 3D coloré par cluster KMeans", "UMAP 3D coloré par compositeur")
)

# Trace clusters
fig_sub.add_trace(
    go.Scatter3d(
        x=X_umap_clusters_3d[:, 0],
        y=X_umap_clusters_3d[:, 1],
        z=X_umap_clusters_3d[:, 2],
        mode='markers',
        marker=dict(size=4, color=clusters, colorscale='Viridis', colorbar=dict(title="Cluster")),
        text=morceaux,
        hovertemplate="Morceau: %{text}<br>Cluster: %{marker.color}<extra></extra>"
    ),
    row=1, col=1
)

# Trace compositeurs
fig_sub.add_trace(
    go.Scatter3d(
        x=X_umap_clusters_3d[:, 0],
        y=X_umap_clusters_3d[:, 1],
        z=X_umap_clusters_3d[:, 2],
        mode='markers',
        marker=dict(size=4, color=le.fit_transform(y), colorscale='Turbo', colorbar=dict(title="Compositeur")),
        text=morceaux,
        hovertemplate="Morceau: %{text}<br>Compositeur: %{marker.color}<extra></extra>"
    ),
    row=1, col=2
)

fig_sub.update_layout(
    width=1200, height=600,
    title_text="UMAP 3D : Cluster KMeans (gauche) vs Compositeur (droite)"
)
#fig_sub.show()
sync_cameras(fig_sub)
#fig.update_traces(marker=dict(size=4))
#fig.show()

KeyboardInterrupt: 

In [21]:

df = pd.DataFrame(data=X_lda_kmeans)
df['Compositeur'] = [i for i in y]
df['Cluster'] = clusters
# Normalisation : chaque case = proportion des oeuvres du compositeur dans le cluster
cluster_composer_counts = df.groupby(['Cluster', 'Compositeur']).size().unstack(fill_value=0)
cluster_composer_counts_norm = cluster_composer_counts.div(oeuvres_par_compositeur, axis=1)
#print(cluster_composer_counts)

#plt.figure(figsize=(18, 18))

fig = px.imshow(
    cluster_composer_counts_norm.values * 100,
    labels=dict(x="Compositeur", y="Cluster", color="Proportion (%)"),
    x=cluster_composer_counts_norm.columns,
    y=cluster_composer_counts_norm.index,
    color_continuous_scale='viridis',
    aspect="auto"
)
fig.update_traces(
    hovertemplate="Cluster: %{y}<br>Compositeur: %{x}<br>Proportion: %{z:.2f}%<extra></extra>"
)
fig.update_layout(
    title="Matrice de confusion (Cluster vs Compositeur) - Proportion (%)",
    xaxis_title="Compositeur",
    yaxis_title="Cluster",
    autosize=False,
    width=1200,
    height=1200
)
fig.show()

### Tentative de détermination d'influences potentielles

In [31]:
from sklearn.cluster import AffinityPropagation
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import plotly.express as px

# 1) Affinity Propagation
ap = AffinityPropagation(damping=0.98392, random_state=12)
clusters_ap = ap.fit_predict(X_scaled)

# 2) Projection LDA en 3D pour visualisation
lda_ap = LDA(n_components=3)
X_lda_ap = lda_ap.fit_transform(X_scaled, y)

# 3) Affichage Plotly
fig = px.scatter_3d(
    x=X_lda_ap[:, 0],
    y=X_lda_ap[:, 1],
    z=X_lda_ap[:, 2],
    color=clusters_ap.astype(str),
    labels={'color': 'Cluster AP'},
    hover_name=morceaux,
    title=f"Nombre de clusters AP : {len(np.unique(clusters_ap))}",
)
fig.update_traces(marker=dict(size=4))
fig.show()

In [32]:
df_conf = pd.DataFrame(data=X_lda_ap, columns=['LD1','LD2','LD3'])
df_conf['Compositeur'] = y
df_conf['Cluster'] = clusters_ap

cluster_composer_counts = df_conf.groupby(['Cluster', 'Compositeur']).size().unstack(fill_value=0)
cluster_composer_counts_norm = cluster_composer_counts.div(oeuvres_par_compositeur, axis=1)

fig = px.imshow(
    cluster_composer_counts_norm.values * 100,
    labels=dict(x="Compositeur", y="Cluster", color="Proportion (%)"),
    x=cluster_composer_counts_norm.columns,
    y=cluster_composer_counts_norm.index,
    color_continuous_scale='viridis',
    aspect="auto"
)
fig.update_traces(
    hovertemplate="Cluster: %{y}<br>Compositeur: %{x}<br>Proportion: %{z:.2f}%<extra></extra>"
)
fig.update_layout(
    title="Matrice de confusion (Cluster vs Compositeur) - Proportion (%)",
    width=1200,
    height=1200
)
fig.show()